# 📄 Google Colab: Live Demo Qwen2.5-VL-3B LoRA (DocVQA Tiếng Việt)
### ⚡ Vận hành trên GPU NVIDIA Tesla T4 (Miễn phí trên Google Colab)

> **Hướng dẫn 1-Click:**
> 1. Vào menu **Runtime** $\rightarrow$ **Change runtime type** $\rightarrow$ Chọn **T4 GPU** $\rightarrow$ Bấm **Save**.
> 2. Bấm tổ hợp phím **`Ctrl + F9`** (hoặc chọn menu **Runtime** $\rightarrow$ **Run all**).
> 3. Chờ khoảng 1.5 phút để hệ thống cài đặt và nạp LoRA Adapter. Kéo xuống dưới cùng để lấy đường link công khai: `Running on public URL: https://xxxx.gradio.live`!

In [ ]:
# [1/4] CÀI ĐẶT CÁC THƯ VIỆN CẦN THIẾT
!pip install -q --no-deps qwen-vl-utils==0.0.8
!pip install -q "transformers>=4.49.0" "peft>=0.13.2" "accelerate>=0.34.2" gradio>=4.0.0 easyocr kaggle

In [ ]:
# [2/4] TẢI BỘ TRỌNG SỐ LORA ADAPTER TỪ KAGGLE (148 MB)
import os, sys, zipfile

os.environ['KAGGLE_USERNAME'] = "lminhsang241"
os.environ['KAGGLE_KEY'] = "KGAT_12612da5e2c3154b6b946bf38c7aed78"

!kaggle datasets download -d lminhsang241/qwen2-5-vl-lora-3b
!unzip -o -q qwen2-5-vl-lora-3b.zip -d lora_adapters

print("✅ Đã tải và giải nén thành công LoRA Adapter!")
!ls -lh lora_adapters

In [ ]:
# [3/4] KHỞI TẠO MÔ HÌNH QWEN2.5-VL-3B VÀ NẠP LORA ADAPTER
import torch, time, re, numpy as np
from PIL import Image, ImageDraw
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info
from peft import PeftModel
import easyocr
import gradio as gr

print(f"🔥 GPU Device: {torch.cuda.get_device_name(0)}")
print(f"🧠 Total VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")

model_name = "Qwen/Qwen2.5-VL-3B-Instruct"
print(f"⏳ Đang nạp Base Model {model_name} (Native FP16)...")
processor = AutoProcessor.from_pretrained(model_name, min_pixels=256*28*28, max_pixels=512*28*28)
base_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    model_name, 
    torch_dtype=torch.float16, 
    device_map="auto"
)

adapter_path = "lora_adapters"
for root, dirs, files in os.walk("lora_adapters"):
    if "adapter_config.json" in files:
        adapter_path = root
        break

print(f"🔗 Đang gắn LoRA Adapter từ {adapter_path}...")
model = PeftModel.from_pretrained(base_model, adapter_path).eval()
print("🎉 Nạp thành công Qwen2.5-VL-3B LoRA (89.63% ANLS) trên GPU Tesla T4!")

print("🔍 Đang nạp EasyOCR Engine hỗ trợ minh chứng...")
reader = easyocr.Reader(['vi', 'en'], gpu=torch.cuda.is_available())
print("✅ Hệ thống đã sẵn sàng phục vụ!")

In [ ]:
# [4/4] KHỞI CHẠY GRADIO WEB UI (CÓ PUBLIC LINK GRADIO.LIVE)
SYSTEM_PROMPT = "Bạn là chuyên gia AI kế toán chuyên đọc và bóc tách hóa đơn, chứng từ tài chính tiếng Việt. Hãy đọc ảnh và trả lời câu hỏi trực tiếp, chính xác, ngắn gọn theo đúng nội dung trên tài liệu, không giải thích lan man."

def predict_docvqa(image, question, enable_bbox):
    if image is None:
        return None, "⚠️ Vui lòng tải lên ảnh hóa đơn hoặc chứng từ.", "0.00s", "0.00 GB"
    if not question or not question.strip():
        question = "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?"
        
    t0 = time.time()
    q_lower = question.lower()
    is_json = any(k in q_lower for k in ["json", "toàn bộ", "cấu trúc", "tất cả", "hạng mục"])
    is_items = any(k in q_lower for k in ["danh sách", "món", "hàng", "dịch vụ", "mặt hàng"])
    max_tokens = 1024 if is_json else (384 if is_items else 160)
    
    # OCR nếu bật BBox
    ocr_results = []
    if enable_bbox and not is_json:
        try:
            img_np = np.array(image.convert("RGB"))
            ocr_results = reader.readtext(img_np)
        except Exception:
            pass
            
    # VLM Inference
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": [{"type": "image", "image": image}, {"type": "text", "text": question.strip()}]}
    ]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(text=[text], images=image_inputs, videos=video_inputs, padding=True, return_tensors="pt").to("cuda")
    
    with torch.no_grad():
        generated_ids = model.generate(**inputs, max_new_tokens=max_tokens, do_sample=False)
        generated_ids_trimmed = [out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)]
        raw_response = processor.batch_decode(generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0].strip()
        
    clean_ans = str(raw_response).strip()
    
    # Grounding Bounding Box
    annotated_img = image.copy().convert("RGB")
    if enable_bbox and not is_json and ocr_results:
        draw = ImageDraw.Draw(annotated_img)
        w, h = annotated_img.size
        cand_digits = re.sub(r'\D', '', clean_ans)
        if len(cand_digits) >= 4:
            for bbox, token_text, conf in ocr_results:
                t_digits = re.sub(r'\D', '', token_text)
                if cand_digits in t_digits or t_digits in cand_digits:
                    pts = np.array(bbox)
                    x1, y1 = int(np.min(pts[:, 0])), int(np.min(pts[:, 1]))
                    x2, y2 = int(np.max(pts[:, 0])), int(np.max(pts[:, 1]))
                    draw.rectangle([max(0, x1-2), max(0, y1-2), min(w, x2+2), min(h, y2+2)], outline="#E11D48", width=3)
                    break
                    
    lat = time.time() - t0
    vram = torch.cuda.memory_allocated() / (1024**3)
    return annotated_img, clean_ans, f"{lat:.2f}s", f"{vram:.2f} GB"

with gr.Blocks(title="Document Visual QA Pro - Qwen2.5-VL", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 📄 Hệ Thống Document Visual Question Answering (DocVQA Pro)")
    gr.Markdown("💡 Mô hình **Qwen2.5-VL-3B LoRA Fine-Tuned (89.63% ANLS trên GPU Tesla T4)**. Tự động trích xuất hóa đơn tiếng Việt & đối soát trực quan.")
    
    with gr.Row():
        with gr.Column(scale=1):
            img_input = gr.Image(type="pil", label="📄 1. Tải lên ảnh Hóa đơn / Chứng từ")
            q_input = gr.Textbox(lines=2, placeholder="Nhập câu hỏi (Ví dụ: Tổng tiền là bao nhiêu?)...", value="Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?", label="💬 2. Câu hỏi cần bóc tách")
            chk_bbox = gr.Checkbox(value=True, label="🎯 Hiển thị Bounding Box đối soát trực quan")
            
            with gr.Row():
                btn_total = gr.Button("💰 Tổng tiền", variant="primary", size="sm")
                btn_items = gr.Button("📦 Danh sách món hàng", size="sm")
                btn_tax = gr.Button("🔢 Mã số thuế", size="sm")
            with gr.Row():
                btn_vendor = gr.Button("🏢 Tên bên bán", size="sm")
                btn_date = gr.Button("📅 Ngày lập", size="sm")
                btn_addr = gr.Button("📍 Địa chỉ", size="sm")
                btn_json = gr.Button("🧾 Trích xuất JSON", size="sm")
            btn_submit = gr.Button("🚀 Phân tích & Trích xuất", variant="primary", size="lg")
            
        with gr.Column(scale=1):
            img_output = gr.Image(type="pil", label="🎯 3. Ảnh Đối Soát Minh Chứng (Bounding Box)")
            txt_output = gr.Textbox(lines=16, label="💬 4. Kết quả Trích xuất từ AI")
            with gr.Row():
                latency_box = gr.Textbox(label="⏱️ Tốc độ suy luận", interactive=False)
                vram_box = gr.Textbox(label="🧠 VRAM sử dụng", interactive=False)
                
    btn_total.click(fn=lambda: "Tổng tiền thanh toán cuối cùng trên hóa đơn là bao nhiêu?", outputs=q_input)
    btn_items.click(fn=lambda: "Danh sách các mặt hàng / dịch vụ được mua trên hóa đơn gồm những gì?", outputs=q_input)
    btn_tax.click(fn=lambda: "Mã số thuế của đơn vị bán hàng trên hóa đơn là gì?", outputs=q_input)
    btn_vendor.click(fn=lambda: "Tên đơn vị / người bán hàng trên hóa đơn là gì?", outputs=q_input)
    btn_date.click(fn=lambda: "Ngày giờ lập hóa đơn là khi nào?", outputs=q_input)
    btn_addr.click(fn=lambda: "Địa chỉ của đơn vị bán hàng là ở đâu?", outputs=q_input)
    btn_json.click(fn=lambda: "Trích xuất toàn bộ thông tin quan trọng của hóa đơn dưới dạng JSON đầy đủ tất cả các trường.", outputs=q_input)
    
    btn_submit.click(fn=predict_docvqa, inputs=[img_input, q_input, chk_bbox], outputs=[img_output, txt_output, latency_box, vram_box])

demo.queue().launch(share=True, debug=True)